# Sigiriya SinLlama legacy RAG comparison v2

In the local `Sigiriya-Hybrid-GraphRAG` directory, build the upload ZIP with
`python -B -m scripts.sinllama_legacy_rag_comparison_v2 bundle`.
Open this notebook in Colab, select **Runtime ? Change runtime type ? T4 GPU**, and run all cells in order. Upload `sigiriya_sinllama_legacy_rag_runtime_v2.zip` from `results/sinllama_legacy_rag_comparison_v2/` when prompted.

A and B reuse the v1 baselines. C applies terminal cleanup to B's completion. D runs Q03's two existing subquestions with B's decoding and their own excerpts. Each answer uses the same SinLlama checkpoint. The final answer uses the strict decision for each unit. The JSON also records the existing experimental verifier's decision.

In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2

In [ ]:
from io import BytesIO
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
print(torch.cuda.get_device_name())
bundle_name = "sigiriya_sinllama_legacy_rag_runtime_v2.zip"
uploaded = files.upload()
assert bundle_name in uploaded, f"Upload {bundle_name}"
workspace = Path(mkdtemp(prefix="sigiriya_legacy_rag_v2_", dir="/content"))
with ZipFile(BytesIO(uploaded[bundle_name])) as archive:
    archive.extractall(workspace)

In [ ]:
import os
import subprocess
import sys

name = "sinllama_legacy_rag_comparison_v2"
output = workspace / "results" / name
subprocess.run(
    [sys.executable, "-B", "-m", f"scripts.{name}", "gpu", "--output", str(output)],
    cwd=workspace, env={**os.environ, "PYTHONIOENCODING": "utf-8"}, check=True,
)

In [ ]:
import json
import pandas as pd

report = json.loads((output / f"{name}_gpu.json").read_text(encoding="utf-8"))
assert {(row["question_id"], row["arm"]) for row in report["results"]} == (
    {(qid, arm) for qid in ("Q01", "Q03", "Q05") for arm in ("A", "B", "C")}
    | {("Q03", "D")}
)
counts = [
    {"arm": arm, "verifier": mode, **totals}
    for arm, modes in report["answer_counts"].items()
    for mode, totals in modes.items()
]
display(pd.DataFrame(counts))

decomposed = next(row for row in report["results"] if row["arm"] == "D")
assert len(decomposed["subquestions"]) == 2
for topic in decomposed["subquestions"]:
    print(topic["heading"], topic["question"])
    print("Raw model output", repr(topic["raw_model_output"]))
    print("Cleaned candidate", repr(topic["cleaned_model_answer"]))
    for mode in ("strict", "experimental"):
        decision = topic[mode]
        print(mode, decision["reason"])
        print("Accepted model answer", decision["verified_answer"])
        print("Extractive fallback", decision["fallback_answer"])
    print()
print("Combined Q03 final answer (strict)")
print(decomposed["final_answer"])

In [ ]:
from shutil import make_archive

archive = make_archive(str(workspace / f"{name}_outputs"), "zip", output)
files.download(archive)